In [7]:
import os
import torch
import pandas as pd
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer
from uuid import uuid5, NAMESPACE_DNS

In [2]:
load_dotenv()

url = os.getenv('QDRANT_URL')
api = os.getenv('QDRANT_API_KEY')

embed_model_name = 'all-MiniLM-L6-v2'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

data_path = 'Data'

In [24]:
data = [i for i in os.listdir(data_path)]
df_QA = pd.read_csv(os.path.join(data_path, data[0]))

combined=''
for name in (df_QA.columns.tolist()):
    combined += df_QA[name] + '. '
    
df_QA['combined'] = combined
df_QA.head()

,qtype,Question,Answer,combined
0,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,LCMV infections can occur after exposure to fr...,susceptibility. Who is at risk for Lymphocytic...
1,symptoms,What are the symptoms of Lymphocytic Choriomen...,LCMV is most commonly recognized as causing ne...,symptoms. What are the symptoms of Lymphocytic...
2,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,Individuals of all ages who come into contact ...,susceptibility. Who is at risk for Lymphocytic...
3,exams and tests,How to diagnose Lymphocytic Choriomeningitis (...,"During the first phase of the disease, the mos...",exams and tests. How to diagnose Lymphocytic C...
4,treatment,What are the treatments for Lymphocytic Chorio...,"Aseptic meningitis, encephalitis, or meningoen...",treatment. What are the treatments for Lymphoc...


In [4]:
client = QdrantClient(
    url=url,
    api_key=api
)

In [5]:
encoder = SentenceTransformer(
    model_name_or_path=embed_model_name, 
    device=device)

client.create_collection(
    collection_name='prac',
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE,
    )
)

True

In [6]:
text = df_QA['combined'].tolist()
embedding = encoder.encode(
    sentences=text,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=(device=='cpu'),
    convert_to_tensor=(device=='cuda'),
    normalize_embeddings=True,
    device=device
)

Batches: 100%|██████████| 129/129 [00:28<00:00,  4.46it/s]


In [11]:
points = []

for idx,(t, emb)  in enumerate(zip(text, embedding)):
    point = PointStruct(
        id=str(uuid5(NAMESPACE_DNS, t)),
        vector=emb.tolist(),
        payload={
            'text': t,
            'qtype': df_QA.iloc[idx]['qtype'],
            'Question': df_QA.iloc[idx]['Question'],
            'Answer': df_QA.iloc[idx]['Answer']
        }
    )
    points.append(point)
    
client.upload_points(
    collection_name='prac',
    points=points,
    batch_size=43,
    parallel=4,
    wait=True,
)
    